# TCC — Análise Empírica do Lido DAO e Liquid Staking no Ethereum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maths0007/TCC_DEFI/blob/master/notebook_tcc_lido_defi.ipynb)

**Autor:** Matheus — TCC Finanças Descentralizadas (DeFi)  
**Recorte Temporal Analisado:** Maio de 2022 a Abril de 2026 (48 Meses / 4 Anos)  

Este Jupyter Notebook executa a pipeline completa de:
1. **Coleta de Dados On-Chain (APIs públicas do DefiLlama)**
2. **Cálculo de Estatísticas Descritivas, Testes Econométricos (t-test / F-test) e Índice HHI**
3. **Geração de TODOS OS 8 GRÁFICOS ACADÊMICOS de Alta Resolução (300 DPI)**


## 1. Instalação e Importação das Dependências

In [ ]:
# Instalar bibliotecas necessárias no Google Colab ou Jupyter local
!pip install -q requests pandas numpy scipy matplotlib seaborn

import os
import sys
import time
import json
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Configuração de visualização inline no notebook
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11
})

print('✓ Todas as dependências carregadas com sucesso!')

## 2. Coleta de Dados On-Chain (APIs Públicas da DefiLlama)

In [ ]:
COINS_URL = 'https://coins.llama.fi'
YIELDS_URL = 'https://yields.llama.fi'
PROTOCOLS_URL = 'https://api.llama.fi'

# Recorte Temporal Solicitado (Maio/2022 a Abril/2026)
DATA_INICIO = pd.to_datetime('2022-05-01')
DATA_FIM = pd.to_datetime('2026-04-30')

def fetch_coins_chart(coin_id, start_ts=1609459200, max_span=500):
    all_prices = []
    start = start_ts
    now = int(datetime.now(timezone.utc).timestamp())
    while start < now:
        url = f'{COINS_URL}/chart/{coin_id}'
        params = {'period': '1d', 'span': max_span, 'start': start}
        resp = requests.get(url, params=params, timeout=60)
        if resp.status_code != 200:
            break
        data = resp.json()
        coins = data.get('coins', {})
        if not coins:
            break
        key = list(coins.keys())[0]
        prices = coins[key].get('prices', [])
        if not prices:
            break
        all_prices.extend(prices)
        start = prices[-1]['timestamp'] + 86400
        time.sleep(0.3)
    return all_prices

print('Buscando preços históricos de ETH e stETH...')
steth_prices = fetch_coins_chart('coingecko:staked-ether')
eth_prices = fetch_coins_chart('coingecko:ethereum')

steth_dict = {datetime.fromtimestamp(p['timestamp'], tz=timezone.utc).strftime('%Y-%m-%d'): p['price'] for p in steth_prices}
eth_dict = {datetime.fromtimestamp(p['timestamp'], tz=timezone.utc).strftime('%Y-%m-%d'): p['price'] for p in eth_prices}

common = sorted(set(steth_dict.keys()) & set(eth_dict.keys()))
records = []
for dt in common:
    s, e = steth_dict[dt], eth_dict[dt]
    if e > 0 and s > 0:
        ratio = s / e
        records.append({
            'data': dt,
            'preco_steth_eth': ratio,
            'preco_steth_usd': s,
            'preco_eth_usd': e,
            'depeg_pct': (ratio - 1.0) * 100
        })

df_depeg = pd.DataFrame(records)
df_depeg['data'] = pd.to_datetime(df_depeg['data'])
df_depeg = df_depeg[(df_depeg['data'] >= DATA_INICIO) & (df_depeg['data'] <= DATA_FIM)].reset_index(drop=True)

# Dataframe de Preço ETH
df_eth_price = pd.DataFrame([{'data': pd.to_datetime(datetime.fromtimestamp(p['timestamp'], tz=timezone.utc).strftime('%Y-%m-%d')), 'preco_eth_usd': p['price']} for p in eth_prices])
df_eth_price = df_eth_price[(df_eth_price['data'] >= DATA_INICIO) & (df_eth_price['data'] <= DATA_FIM)].reset_index(drop=True)

# Coleta de APY Lido e Solo Staking
print('Buscando APR/APY do Lido stETH...')
pools_resp = requests.get(f'{YIELDS_URL}/pools', timeout=60).json().get('data', [])
lido_pool = [p for p in pools_resp if 'lido' in p.get('project', '').lower() and p.get('chain', '').lower() == 'ethereum' and 'STETH' in p.get('symbol', '').upper()]
pool_uuid = lido_pool[0]['pool'] if lido_pool else '747c1d2a-c668-4682-b9f9-296708a3dd90'

hist_resp = requests.get(f'{YIELDS_URL}/chart/{pool_uuid}', timeout=60).json().get('data', [])
df_apr_lido = pd.DataFrame([{'data': e.get('timestamp', '')[:10], 'apy_pct': e.get('apy')} for e in hist_resp]).dropna()
df_apr_lido['data'] = pd.to_datetime(df_apr_lido['data'])
df_apr_lido = df_apr_lido[(df_apr_lido['data'] >= DATA_INICIO) & (df_apr_lido['data'] <= DATA_FIM)].reset_index(drop=True)

df_apr_direto = df_apr_lido.copy()
df_apr_direto['apr_direto_estimado_pct'] = df_apr_direto['apy_pct'] / 0.90
df_apr_direto['spread_pct'] = df_apr_direto['apr_direto_estimado_pct'] - df_apr_direto['apy_pct']

# Coleta de TVL Lido
print('Buscando TVL da Lido...')
lido_data = requests.get(f'{PROTOCOLS_URL}/protocol/lido', timeout=60).json()
tvl_eth = lido_data.get('chainTvls', {}).get('Ethereum', {}).get('tvl', [])
df_tvl = pd.DataFrame([{'data': datetime.fromtimestamp(e['date'], tz=timezone.utc).strftime('%Y-%m-%d'), 'tvl_total_usd': e['totalLiquidityUSD']} for e in tvl_eth])
df_tvl['data'] = pd.to_datetime(df_tvl['data'])
df_tvl = df_tvl[(df_tvl['data'] >= DATA_INICIO) & (df_tvl['data'] <= DATA_FIM)].reset_index(drop=True)

# Coleta de Market Share Histórico
print('Buscando histórico de Market Share dos LSDs...')
top_lsds = ['lido', 'rocket-pool', 'coinbase-wrapped-staked-eth', 'binance-staked-eth', 'frax-ether', 'stakewise', 'stader', 'swell', 'ankr']
all_hist = {}
for slug in top_lsds:
    try:
        r_prot = requests.get(f'{PROTOCOLS_URL}/protocol/{slug}', timeout=30)
        if r_prot.status_code == 200:
            d_prot = r_prot.json()
            tvl_lst = d_prot.get('chainTvls', {}).get('Ethereum', {}).get('tvl', []) or d_prot.get('tvl', [])
            for e in tvl_lst:
                dt_str = datetime.fromtimestamp(e.get('date', 0), tz=timezone.utc).strftime('%Y-%m-%d')
                if dt_str not in all_hist:
                    all_hist[dt_str] = {}
                all_hist[dt_str][slug] = e.get('totalLiquidityUSD', 0)
    except Exception:
        pass
    time.sleep(0.2)

rows_share = []
for dt_str in sorted(all_hist.keys()):
    r_dict = {'data': dt_str}
    tot = sum(all_hist[dt_str].values())
    r_dict['total_top_tvl'] = tot
    for slug in top_lsds:
        r_dict[f'tvl_{slug.replace("-", "_")}'] = all_hist[dt_str].get(slug, 0)
    if tot > 0 and 'lido' in all_hist[dt_str]:
        r_dict['lido_share_pct'] = (all_hist[dt_str]['lido'] / tot) * 100
    rows_share.append(r_dict)

df_share_hist = pd.DataFrame(rows_share)
df_share_hist['data'] = pd.to_datetime(df_share_hist['data'])
df_share_hist = df_share_hist[(df_share_hist['data'] >= DATA_INICIO) & (df_share_hist['data'] <= DATA_FIM)].reset_index(drop=True)

print('✓ Todos os dados on-chain foram coletados e recortados (Maio/2022 a Abril/2026)!')

## 3. Estatísticas Descritivas e Testes Econométricos

In [ ]:
depeg = df_depeg['depeg_pct']
ratio = df_depeg['preco_steth_eth']

print('=' * 65)
print('   ESTATÍSTICAS DESCRITIVAS DO DEPEG (MAIO/2022 A ABRIL/2026)')
print('=' * 65)
print(f'Total de Dias Analisados:      {len(depeg)}')
print(f'Preço Médio (stETH/ETH):        {ratio.mean():.6f}')
print(f'Depeg Médio (%):                {depeg.mean():.4f}%')
print(f'Depeg Mediano (%):              {depeg.median():.4f}%')
print(f'Desvio Padrão (%):              {depeg.std():.4f}%')
print(f'Mínimo (Maior Desconto):        {depeg.min():.4f}% (Crash Terra/LUNA)')
print(f'Máximo (Maior Prêmio):          {depeg.max():.4f}% (Pós-FTX)')
print(f'Assimetria (Skewness):          {stats.skew(depeg):.4f}')
print(f'Curtose (Kurtosis):             {stats.kurtosis(depeg):.4f}')
print(f'Autocorrelação AR(1):           {depeg.autocorr(lag=1):.4f}')
print(f'Autocorrelação AR(5):           {depeg.autocorr(lag=5):.4f}')
print('=' * 65)

# Teste Pré vs Pós Shanghai
data_shanghai = pd.to_datetime('2023-04-12')
pre = df_depeg[df_depeg['data'] <= data_shanghai]['depeg_pct']
pos = df_depeg[df_depeg['data'] > data_shanghai]['depeg_pct']

t_stat, p_val_t = stats.ttest_ind(pre, pos, equal_var=False)
reducao_vol = (1 - pos.std() / pre.std()) * 100

print('\n' + '=' * 65)
print('   TESTE ECONOMÉTRICO: IMPACTO DO SHANGHAI UPGRADE (12/04/2023)')
print('=' * 65)
print(f'Pré-Shanghai (Maio/22 - Abr/23): Média = {pre.mean():.4f}% | Desv.Pad = {pre.std():.4f}%')
print(f'Pós-Shanghai (Abr/23 - Abr/26): Média = {pos.mean():.4f}% | Desv.Pad = {pos.std():.4f}%')
print(f'Redução na Volatilidade do Peg: {reducao_vol:.1f}%')
print(f'Teste t de Student (p-value):   {p_val_t:.4e}')
print('=' * 65)

## 4. Visualizações Acadêmicas (Todos os 8 Gráficos)

### Figura 1: Série Histórica da Paridade stETH/ETH e Depeg %

In [ ]:
# Figura 1
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1.2]})

ax1.plot(df_depeg['data'], df_depeg['preco_steth_eth'], color='#1f77b4', linewidth=1.2, label='Razão stETH/ETH')
ax1.axhline(1.0, color='#d62728', linestyle='--', linewidth=1.2, label='Paridade 1:1 (ETH)')

eventos = [
    (pd.to_datetime('2022-05-12'), 0.93, 'Crash Terra/LUNA', '#d62728'),
    (pd.to_datetime('2022-11-09'), 0.98, 'Falência FTX', '#ff7f0e'),
    (pd.to_datetime('2023-04-12'), 1.00, 'Shanghai Upgrade', '#2ca02c'),
    (pd.to_datetime('2026-04-18'), 0.99, 'Hack KelpDAO', '#9467bd')
]
for dt, y_pos, texto, cor in eventos:
    if df_depeg['data'].min() <= dt <= df_depeg['data'].max():
        ax1.axvline(dt, color=cor, linestyle=':', alpha=0.7, linewidth=1.2)
        ax1.annotate(texto, xy=(dt, y_pos), xytext=(dt + pd.Timedelta(days=20), y_pos - 0.03),
                     arrowprops=dict(arrowstyle='->', color=cor, lw=0.9), fontsize=8, color=cor, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=cor, alpha=0.8))

ax1.set_ylabel('Preço stETH em ETH')
ax1.set_title('Figura 1: Série Histórica da Paridade stETH/ETH (Maio/2022 – Abril/2026)', fontweight='bold', pad=12)
ax1.legend(loc='lower right', frameon=True)
ax1.set_ylim(0.85, 1.08)

ax2.plot(df_depeg['data'], df_depeg['depeg_pct'], color='#9467bd', linewidth=1.0, label='Depeg (%)')
ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)
ax2.axhline(-2.0, color='#ff7f0e', linestyle=':', linewidth=1.0, label='Limiar Alerta (-2%)')
ax2.axhline(-5.0, color='#d62728', linestyle='--', linewidth=1.0, label='Depeg Severo (-5%)')
ax2.fill_between(df_depeg['data'], df_depeg['depeg_pct'], 0, where=(df_depeg['depeg_pct'] < 0), color='#d62728', alpha=0.2)
ax2.fill_between(df_depeg['data'], df_depeg['depeg_pct'], 0, where=(df_depeg['depeg_pct'] >= 0), color='#2ca02c', alpha=0.2)

ax2.set_ylabel('Depeg (%)')
ax2.set_xlabel('Data')
ax2.legend(loc='lower right', frameon=True)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))

plt.tight_layout()
plt.show()

### Figura 2: Impacto da Habilitação de Saques Nativos (Shanghai Upgrade)

In [ ]:
# Figura 2
df_depeg['periodo'] = np.where(df_depeg['data'] <= data_shanghai, 'Pré-Shanghai\n(Sem Saques)', 'Pós-Shanghai\n(Com Saques)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [2, 1]})

pre_data = df_depeg[df_depeg['data'] <= data_shanghai]
pos_data = df_depeg[df_depeg['data'] > data_shanghai]

ax1.plot(pre_data['data'], pre_data['depeg_pct'], color='#d62728', linewidth=1.0, label='Pré-Shanghai (Volatilidade Alta)')
ax1.plot(pos_data['data'], pos_data['depeg_pct'], color='#2ca02c', linewidth=1.0, label='Pós-Shanghai (Estabilizado)')
ax1.axvline(data_shanghai, color='black', linestyle='--', linewidth=1.5, label='Shanghai (12/04/2023)')
ax1.set_ylabel('Depeg (%)')
ax1.set_title('A. Evolução Temporal do Depeg (%)', fontweight='bold')
ax1.legend(loc='lower right')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))

sns.boxplot(x='periodo', y='depeg_pct', data=df_depeg, ax=ax2, hue='periodo', palette=['#ff9999', '#99ff99'], width=0.4, legend=False)
ax2.set_title('B. Volatilidade Comparativa', fontweight='bold')
ax2.set_ylabel('Depeg (%)')
ax2.set_xlabel('')

plt.tight_layout()
plt.show()

### Figura 3: Histograma e Densidade KDE da Distribuição do Depeg (%)

In [ ]:
# Figura 3
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(depeg, kde=True, ax=ax, color='#1f77b4', bins=60, stat='density', alpha=0.4, edgecolor='white')

ax.axvline(depeg.mean(), color='#d62728', linestyle='--', linewidth=1.2, label=f'Média: {depeg.mean():.2f}%')
ax.axvline(depeg.median(), color='#2ca02c', linestyle='-', linewidth=1.2, label=f'Mediana: {depeg.median():.2f}%')
ax.axvline(np.percentile(depeg, 1), color='#ff7f0e', linestyle=':', linewidth=1.2, label=f'Percentil 1%: {np.percentile(depeg, 1):.2f}%')

ax.set_xlabel('Depeg (%)')
ax.set_ylabel('Densidade de Frequência')
ax.set_title('Figura 3: Distribuição Empírica do Depeg stETH/ETH (Maio/2022 – Abril/2026)', fontweight='bold', pad=12)
ax.set_xlim(-15, 5)
ax.legend(loc='upper left', frameon=True)

plt.tight_layout()
plt.show()

### Figura 4: Rendimento de Staking — Lido stETH vs. Solo Staking Direto

In [ ]:
# Figura 4
fig, ax = plt.subplots(figsize=(10, 5))
merged_apr = pd.merge(df_apr_lido, df_apr_direto[['data', 'apr_direto_estimado_pct']], on='data').dropna()

ax.plot(merged_apr['data'], merged_apr['apr_direto_estimado_pct'], color='#1f77b4', linewidth=1.5, label='Solo Staking Direto (Bruto Estimado)')
ax.plot(merged_apr['data'], merged_apr['apy_pct'], color='#2ca02c', linewidth=1.5, label='Lido stETH (Líquido pós-taxa de 10%)')
ax.fill_between(merged_apr['data'], merged_apr['apy_pct'], merged_apr['apr_direto_estimado_pct'], color='#ff7f0e', alpha=0.3, label='Spread da Comissão Lido DAO (10%)')

ax.set_ylabel('Taxa Anual de Rendimento (%)')
ax.set_xlabel('Data')
ax.set_title('Figura 4: Rendimento de Staking — Lido stETH vs. Solo Staking (Maio/2022 – Abril/2026)', fontweight='bold', pad=12)
ax.legend(loc='upper right', frameon=True)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))

plt.tight_layout()
plt.show()

### Figura 5: TVL do Lido (USD) vs. Cotação do Ethereum (USD)

In [ ]:
# Figura 5
fig, ax1 = plt.subplots(figsize=(11, 5.5))
merged_tvl = pd.merge(df_tvl, df_eth_price, on='data').dropna()
merged_tvl['tvl_bilhoes'] = merged_tvl['tvl_total_usd'] / 1e9

color_tvl = '#1f77b4'
ax1.plot(merged_tvl['data'], merged_tvl['tvl_bilhoes'], color=color_tvl, linewidth=1.5, label='TVL Lido (Bilhões USD)')
ax1.fill_between(merged_tvl['data'], merged_tvl['tvl_bilhoes'], color=color_tvl, alpha=0.15)
ax1.set_ylabel('TVL Lido (Bilhões de USD)', color=color_tvl, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color_tvl)

ax2 = ax1.twinx()
color_eth = '#ff7f0e'
ax2.plot(merged_tvl['data'], merged_tvl['preco_eth_usd'], color=color_eth, linewidth=1.2, linestyle='--', label='Preço ETH/USD')
ax2.set_ylabel('Preço do Ethereum (USD)', color=color_eth, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color_eth)

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
ax1.set_title('Figura 5: Relação entre o Capital Trava no Lido (TVL) e o Preço do ETH (Maio/2022 – Abril/2026)', fontweight='bold', pad=12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=True)

plt.tight_layout()
plt.show()

### Figura 6: Evolução do Market Share dos Protocolos LSD/LRT (Área Empilhada)

In [ ]:
# Figura 6
fig, ax = plt.subplots(figsize=(11, 6))
cols_tvl = [c for c in df_share_hist.columns if c.startswith('tvl_')]
df_clean = df_share_hist.dropna(subset=['total_top_tvl']).copy()
df_clean = df_clean[df_clean['total_top_tvl'] > 0]

nomes_map = {
    'tvl_lido': 'Lido DAO',
    'tvl_binance_staked_eth': 'Binance (bETH)',
    'tvl_coinbase_wrapped_staked_eth': 'Coinbase (cbETH)',
    'tvl_rocket_pool': 'Rocket Pool (rETH)',
    'tvl_frax_ether': 'Frax Ether (frxETH)',
    'tvl_stakewise': 'StakeWise',
    'tvl_stader': 'Stader (ETHx)',
    'tvl_swell': 'Swell (swETH)',
    'tvl_ankr': 'Ankr'
}

ultimos = df_clean[cols_tvl].iloc[-1].fillna(0)
cols_ordenadas = ultimos.sort_values(ascending=False).index.tolist()

y_data = []
labels = []
for c in cols_ordenadas:
    s = (df_clean[c].fillna(0) / df_clean['total_top_tvl'] * 100).values
    y_data.append(s)
    labels.append(nomes_map.get(c, c.replace('tvl_', '').title()))

colors = sns.color_palette('tab10', len(labels))
ax.stackplot(df_clean['data'], y_data, labels=labels, colors=colors, alpha=0.85)
ax.axhline(33.33, color='black', linestyle='--', linewidth=1.5, label='Teto Crítico de Consenso (33,3%)')

ax.set_ylabel('Participação de Mercado - Market Share (%)')
ax.set_xlabel('Data')
ax.set_title('Figura 6: Evolução do Market Share dos Protocolos LSD/LRT (Maio/2022 – Abril/2026)', fontweight='bold', pad=12)
ax.set_ylim(0, 100)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=True)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))

plt.tight_layout()
plt.show()

### Figura 7: Diagrama de Regressão: Concentração da Lido DAO vs. Depeg stETH

In [ ]:
# Figura 7
merged_scat = pd.merge(df_share_hist[['data', 'lido_share_pct']], df_depeg[['data', 'depeg_pct']], on='data').dropna()

fig, ax = plt.subplots(figsize=(8.5, 5.5))
sns.regplot(x='lido_share_pct', y='depeg_pct', data=merged_scat, ax=ax,
            color='#1f77b4', scatter_kws={'alpha': 0.3, 's': 15},
            line_kws={'color': '#d62728', 'linewidth': 2, 'label': 'Regressão OLS'})

ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.axvline(33.33, color='black', linestyle='--', linewidth=1.0, label='Teto Descentralização (33.3%)')
ax.set_xlabel('Market Share da Lido DAO (%)')
ax.set_ylabel('Depeg stETH (%)')
ax.set_title('Figura 7: Regressão OLS: Concentração do Lido DAO vs. Magnitude do Depeg', fontweight='bold', pad=12)
ax.legend(loc='lower left', frameon=True)

plt.tight_layout()
plt.show()

### Figura 8: Série do Índice Herfindahl-Hirschman (HHI) de Concentração de Mercado

In [ ]:
# Figura 8
hhi_values = []
for idx, row in df_clean.iterrows():
    total = row['total_top_tvl']
    shares = [(row[c] / total * 100) for c in cols_tvl if pd.notna(row[c])]
    hhi_values.append(sum(s ** 2 for s in shares))

df_clean['hhi'] = hhi_values

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_clean['data'], df_clean['hhi'], color='#d62728', linewidth=1.5, label='Índice HHI (Liquid Staking)')
ax.axhline(2500, color='#ff7f0e', linestyle='--', linewidth=1.2, label='Altamente Concentrado (HHI > 2500)')
ax.axhline(1500, color='#2ca02c', linestyle=':', linewidth=1.2, label='Moderadamente Concentrado (1500-2500)')
ax.fill_between(df_clean['data'], df_clean['hhi'], 2500, where=(df_clean['hhi'] >= 2500), color='#d62728', alpha=0.15)

ax.set_ylabel('Índice Herfindahl-Hirschman (HHI)')
ax.set_xlabel('Data')
ax.set_title('Figura 8: Evolução da Concentração de Mercado no Ecossistema Liquid Staking (HHI)', fontweight='bold', pad=12)
ax.set_ylim(0, 10000)
ax.legend(loc='upper right', frameon=True)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))

plt.tight_layout()
plt.show()